# CS5224 Data Curation — Stage 1B

## Add source metadata and permanent source record IDs

Stage 1A verified the three raw retailer workbooks and established a common raw schema.

Stage 1B performs the full-row transformation for all source records.

For every source record, the following provenance fields will be added:

- `retailer`
- `source_file`
- `source_sheet`
- `source_row_number`
- `source_record_id`

The original source values will not be cleaned or interpreted at this stage.

Expected input row counts:

- RedMart: **97,779**
- FairPrice: **69,402**
- Cold Storage: **10,457**
- Total: **177,638**

The raw Excel workbooks remain unchanged.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import pandas as pd

# Notebook location:
# CartOptix/CartOptix/scripts

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
STAGING_DATA_DIR = PROJECT_ROOT / "data" / "staging"

# Create the staging directory if it does not already exist.
STAGING_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Current notebook folder:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nRaw data directory:")
print(RAW_DATA_DIR)

print("\nStaging data directory:")
print(STAGING_DATA_DIR)


Current notebook folder:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\scripts

Project root:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix

Raw data directory:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\raw

Staging data directory:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging


## 2. Define source files and worksheet names

In [2]:
SOURCE_CONFIG = {
    "redmart": {
        "file": "redmart_all_products_final.xlsx",
        "sheet": "Sheet1",
        "expected_rows": 97_779,
    },
    "fairprice": {
        "file": "fairprice_all_products_final.xlsx",
        "sheet": "FairPrice Products",
        "expected_rows": 69_402,
    },
    "coldstorage": {
        "file": "coldstorage_all_products_final.xlsx",
        "sheet": "Cold Storage Products",
        "expected_rows": 10_457,
    },
}

for retailer, config in SOURCE_CONFIG.items():
    file_path = RAW_DATA_DIR / config["file"]

    print(f"{retailer:12s} -> {file_path}")
    print(f"Exists: {file_path.exists()}")

    assert file_path.exists(), f"Missing source file: {file_path}"


redmart      -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\raw\redmart_all_products_final.xlsx
Exists: True
fairprice    -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\raw\fairprice_all_products_final.xlsx
Exists: True
coldstorage  -> D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\raw\coldstorage_all_products_final.xlsx
Exists: True


## 3. Define the common raw schema

In [3]:
COMMON_RAW_FIELDS = [
    "source_category_raw",
    "product_name_raw",
    "product_price_raw",
    "original_price_raw",
    "specification_raw",
    "savings_raw",
    "stock_raw",
    "offer_raw",
    "image_url_raw",
    "product_url_raw",
    "comments_raw",
    "brand_raw",
]

print("Common raw fields:")
for i, field in enumerate(COMMON_RAW_FIELDS, start=1):
    print(f"{i:2d}. {field}")


Common raw fields:
 1. source_category_raw
 2. product_name_raw
 3. product_price_raw
 4. original_price_raw
 5. specification_raw
 6. savings_raw
 7. stock_raw
 8. offer_raw
 9. image_url_raw
10. product_url_raw
11. comments_raw
12. brand_raw


## 4. Define source-column mappings

In [4]:
COLUMN_MAPS = {
    "redmart": {
        "Keyword": "source_category_raw",
        "Product Name": "product_name_raw",
        "Product Price": "product_price_raw",
        "Original Price": "original_price_raw",
        "Specification": "specification_raw",
        "Savings": "savings_raw",
        "in-stock": "stock_raw",
        "offer": "offer_raw",
        "Image URL": "image_url_raw",
        "product url": "product_url_raw",
        "other comments": "comments_raw",
        "brand": "brand_raw",
    },

    "fairprice": {
        "Keyword": "source_category_raw",
        "Product Name": "product_name_raw",
        "Product Price": "product_price_raw",
        "Original Price": "original_price_raw",
        "Specification": "specification_raw",
        "Savings": "savings_raw",
        "in-stock": "stock_raw",
        "offer": "offer_raw",
        "Image URL": "image_url_raw",
        "product url": "product_url_raw",
        "other comments": "comments_raw",
        "Brand": "brand_raw",
    },

    "coldstorage": {
        "Keyword": "source_category_raw",
        "Product Name": "product_name_raw",
        "Product Price": "product_price_raw",
        "Original Price": "original_price_raw",
        "Specification": "specification_raw",
        "Savings": "savings_raw",
        "in-stock": "stock_raw",
        "offer": "offer_raw",
        "Image URL": "image_url_raw",
        "product url": "product_url_raw",
        "Brand": "brand_raw",
    },
}


## 5. Load the three complete Excel workbooks

In [5]:
raw_dataframes = {}

for retailer, config in SOURCE_CONFIG.items():
    file_path = RAW_DATA_DIR / config["file"]

    print(f"Loading {retailer}...")

    df = pd.read_excel(
        file_path,
        sheet_name=config["sheet"],
        engine="openpyxl"
    )

    raw_dataframes[retailer] = df

    print(
        f"{retailer:12s}: "
        f"{len(df):,} rows x {len(df.columns)} columns"
    )


Loading redmart...
redmart     : 97,779 rows x 12 columns
Loading fairprice...
fairprice   : 69,402 rows x 12 columns
Loading coldstorage...
coldstorage : 10,457 rows x 11 columns


## 6. Verify source row counts

In [6]:
print("Source row-count verification")
print("-" * 50)

for retailer, config in SOURCE_CONFIG.items():
    actual = len(raw_dataframes[retailer])
    expected = config["expected_rows"]

    print(
        f"{retailer:12s}: "
        f"actual={actual:>8,} | expected={expected:>8,}"
    )

    assert actual == expected, (
        f"{retailer}: expected {expected:,} rows "
        f"but loaded {actual:,}"
    )

print("\nPASS: All source row counts match Stage 1A.")


Source row-count verification
--------------------------------------------------
redmart     : actual=  97,779 | expected=  97,779
fairprice   : actual=  69,402 | expected=  69,402
coldstorage : actual=  10,457 | expected=  10,457

PASS: All source row counts match Stage 1A.


## 7. Verify original columns before renaming

In [7]:
for retailer, df in raw_dataframes.items():
    source_columns = set(df.columns)
    expected_columns = set(COLUMN_MAPS[retailer].keys())

    unmapped_source_columns = source_columns - expected_columns
    missing_source_columns = expected_columns - source_columns

    print("=" * 70)
    print(retailer.upper())

    print(
        "Unmapped source columns:",
        sorted(unmapped_source_columns)
        if unmapped_source_columns
        else "None"
    )

    print(
        "Expected columns missing from source:",
        sorted(missing_source_columns)
        if missing_source_columns
        else "None"
    )

    assert not unmapped_source_columns, (
        f"{retailer} contains unmapped columns: "
        f"{unmapped_source_columns}"
    )

    assert not missing_source_columns, (
        f"{retailer} is missing expected columns: "
        f"{missing_source_columns}"
    )

print("\nPASS: All source columns match the Stage 1A mappings.")


REDMART
Unmapped source columns: None
Expected columns missing from source: None
FAIRPRICE
Unmapped source columns: None
Expected columns missing from source: None
COLDSTORAGE
Unmapped source columns: None
Expected columns missing from source: None

PASS: All source columns match the Stage 1A mappings.


## 8. Define the Stage 1B transformation function

In [8]:
def prepare_stage1b_dataframe(retailer, df, config):
    """
    Convert one retailer DataFrame into the common raw schema
    and attach source provenance metadata.

    No data cleaning is performed in Stage 1B.
    """

    result = df.copy()

    # ---------------------------------------------------------
    # 1. Add source provenance metadata
    # ---------------------------------------------------------

    result["retailer"] = retailer
    result["source_file"] = config["file"]
    result["source_sheet"] = config["sheet"]

    # Excel row 1 contains headers.
    # Therefore DataFrame row 0 corresponds to Excel row 2.
    result["source_row_number"] = range(
        2,
        len(result) + 2
    )

    # Deterministic source record ID.
    result["source_record_id"] = (
        retailer
        + "_"
        + result["source_row_number"]
            .astype(str)
            .str.zfill(8)
    )

    # ---------------------------------------------------------
    # 2. Rename original source columns
    # ---------------------------------------------------------

    result = result.rename(
        columns=COLUMN_MAPS[retailer]
    )

    # ---------------------------------------------------------
    # 3. Add any common raw fields that the retailer lacks
    # ---------------------------------------------------------

    for field in COMMON_RAW_FIELDS:
        if field not in result.columns:
            result[field] = pd.NA

    # ---------------------------------------------------------
    # 4. Put all columns into one consistent order
    # ---------------------------------------------------------

    metadata_columns = [
        "retailer",
        "source_file",
        "source_sheet",
        "source_row_number",
        "source_record_id",
    ]

    result = result[
        metadata_columns + COMMON_RAW_FIELDS
    ]

    return result


## 9. Transform each retailer

In [9]:
stage1b_dataframes = {}

for retailer, df in raw_dataframes.items():
    config = SOURCE_CONFIG[retailer]

    transformed = prepare_stage1b_dataframe(
        retailer=retailer,
        df=df,
        config=config
    )

    stage1b_dataframes[retailer] = transformed

    print(
        f"{retailer:12s}: "
        f"{transformed.shape[0]:,} rows x "
        f"{transformed.shape[1]} columns"
    )


redmart     : 97,779 rows x 17 columns
fairprice   : 69,402 rows x 17 columns
coldstorage : 10,457 rows x 17 columns


## 10. Inspect the final Stage 1B column structure

In [10]:
print("Stage 1B columns:")
print("-" * 50)

for i, column in enumerate(
    stage1b_dataframes["redmart"].columns,
    start=1
):
    print(f"{i:2d}. {column}")


Stage 1B columns:
--------------------------------------------------
 1. retailer
 2. source_file
 3. source_sheet
 4. source_row_number
 5. source_record_id
 6. source_category_raw
 7. product_name_raw
 8. product_price_raw
 9. original_price_raw
10. specification_raw
11. savings_raw
12. stock_raw
13. offer_raw
14. image_url_raw
15. product_url_raw
16. comments_raw
17. brand_raw


## 11. Inspect sample transformed records

In [11]:
display(stage1b_dataframes["redmart"].head(3))


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,product_name_raw,product_price_raw,original_price_raw,specification_raw,savings_raw,stock_raw,offer_raw,image_url_raw,product_url_raw,comments_raw,brand_raw
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,12.0,NaN,6 × 100 g,NaN,Yes,No,https://sg-live-01.slatic.net/p/5eaa6087b6c250...,https://www.lazada.sg/products/pdp-i303336401-...,NaN,Sojasun
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,10.9,NaN,4 × 125 g,Spend $45 + Free Gift,Yes,Yes,https://sg-test-11.slatic.net/p/9f86cfbce0b472...,https://www.lazada.sg/products/pdp-i462726209-...,NaN,Monoprix
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,shop-chilled-desserts,Top Gourmet Cheng Tng,3.0,NaN,400 g,NaN,Yes,No,https://sg-live-01.slatic.net/p/7579bbca2beaa0...,https://www.lazada.sg/products/pdp-i303250731-...,NaN,Top Gourmet


In [12]:
display(stage1b_dataframes["fairprice"].head(3))


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,product_name_raw,product_price_raw,original_price_raw,specification_raw,savings_raw,stock_raw,offer_raw,image_url_raw,product_url_raw,comments_raw,brand_raw
0,fairprice,fairprice_all_products_final.xlsx,FairPrice Products,2,fairprice_00000002,Organic Snacks,Yifon Roasted Organic Chestnuts,1.68,NaN,100g,Buy 2 At $2.45,Yes,Yes,https://media.nedigital.sg/fairprice/fpol/medi...,https://www.fairprice.com.sg/product/yifon-roa...,NaN,Yifon
1,fairprice,fairprice_all_products_final.xlsx,FairPrice Products,3,fairprice_00000003,Organic Snacks,Sun-Maid Organic Raisins - Mini-Snacks,5.45,NaN,170g (12 per pack),NaN,Yes,No,https://media.nedigital.sg/fairprice/fpol/medi...,https://www.fairprice.com.sg/product/13106097,NaN,Sun-Maid
2,fairprice,fairprice_all_products_final.xlsx,FairPrice Products,4,fairprice_00000004,Organic Snacks,Yes Natural Organic Brown Rice Rings - Cheese,2.50,NaN,40g,NaN,Yes,No,https://media.nedigital.sg/fairprice/fpol/medi...,https://www.fairprice.com.sg/product/yes-natur...,NaN,Yesnatural


In [13]:
display(stage1b_dataframes["coldstorage"].head(3))


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,product_name_raw,product_price_raw,original_price_raw,specification_raw,savings_raw,stock_raw,offer_raw,image_url_raw,product_url_raw,comments_raw,brand_raw
0,coldstorage,coldstorage_all_products_final.xlsx,Cold Storage Products,2,coldstorage_00000002,100% Aussie Goodness: Freshest Picks,Australia Pork Prime Ribs 400g,12.68,NaN,400g,NaN,Yes,No,https://coldstorage.com.sg/_next/image?url=htt...,https://coldstorage.com.sg/product/australia-p...,<NA>,*
1,coldstorage,coldstorage_all_products_final.xlsx,Cold Storage Products,3,coldstorage_00000003,100% Aussie Goodness: Freshest Picks,Celery Australia 1kg,4.90,NaN,1kg,NaN,Yes,No,https://coldstorage.com.sg/_next/image?url=htt...,https://coldstorage.com.sg/product/celery-aust...,<NA>,*
2,coldstorage,coldstorage_all_products_final.xlsx,Cold Storage Products,4,coldstorage_00000004,100% Aussie Goodness: Freshest Picks,Laurent Finest Pane Di Casa Sesame Bread 1s,9.35,NaN,1s,NaN,Yes,No,https://coldstorage.com.sg/_next/image?url=htt...,https://coldstorage.com.sg/product/laurent-fin...,<NA>,LAURENT


## 12. Verify source row numbering

In [14]:
for retailer, df in stage1b_dataframes.items():
    print(retailer)

    print(
        "First source row:",
        df["source_row_number"].iloc[0]
    )

    print(
        "Last source row:",
        df["source_row_number"].iloc[-1]
    )

    print(
        "Number of records:",
        len(df)
    )

    print()


redmart
First source row: 2
Last source row: 97780
Number of records: 97779

fairprice
First source row: 2
Last source row: 69403
Number of records: 69402

coldstorage
First source row: 2
Last source row: 10458
Number of records: 10457



## 13. Verify source IDs are unique within each retailer

In [15]:
for retailer, df in stage1b_dataframes.items():
    duplicate_ids = (
        df["source_record_id"]
        .duplicated()
        .sum()
    )

    print(
        f"{retailer:12s}: "
        f"duplicate source_record_id = {duplicate_ids}"
    )

    assert duplicate_ids == 0

print("\nPASS: No duplicate source IDs within any retailer.")


redmart     : duplicate source_record_id = 0
fairprice   : duplicate source_record_id = 0
coldstorage : duplicate source_record_id = 0

PASS: No duplicate source IDs within any retailer.


## 14. Combine all three retailers

In [16]:
stage1b_all = pd.concat(
    [
        stage1b_dataframes["redmart"],
        stage1b_dataframes["fairprice"],
        stage1b_dataframes["coldstorage"],
    ],
    ignore_index=True
)

print("Combined Stage 1B shape:")
print(stage1b_all.shape)


Combined Stage 1B shape:
(177638, 17)


## 15. Verify the combined row count

In [17]:
expected_total = sum(
    config["expected_rows"]
    for config in SOURCE_CONFIG.values()
)

actual_total = len(stage1b_all)

print("Expected total:", f"{expected_total:,}")
print("Actual total:  ", f"{actual_total:,}")

assert actual_total == expected_total

print("\nPASS: Combined row count is correct.")


Expected total: 177,638
Actual total:   177,638

PASS: Combined row count is correct.


## 16. Verify source IDs are globally unique

In [18]:
duplicate_source_ids = (
    stage1b_all["source_record_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate source_record_id values:",
    duplicate_source_ids
)

assert duplicate_source_ids == 0

print(
    "PASS: Every Stage 1B source record "
    "has a unique source_record_id."
)


Duplicate source_record_id values: 0
PASS: Every Stage 1B source record has a unique source_record_id.


## 17. Verify retailer counts after concatenation

In [19]:
retailer_counts = (
    stage1b_all["retailer"]
    .value_counts()
)

print(retailer_counts)

for retailer, config in SOURCE_CONFIG.items():
    actual = (
        stage1b_all["retailer"] == retailer
    ).sum()

    expected = config["expected_rows"]

    assert actual == expected

print("\nPASS: Retailer counts remain unchanged.")


retailer
redmart        97779
fairprice      69402
coldstorage    10457
Name: count, dtype: int64

PASS: Retailer counts remain unchanged.


## 18. Verify provenance metadata is complete

In [20]:
metadata_columns = [
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",
]

missing_metadata = (
    stage1b_all[metadata_columns]
    .isna()
    .sum()
)

print(missing_metadata)

assert missing_metadata.sum() == 0

print("\nPASS: No provenance metadata is missing.")


retailer             0
source_file          0
source_sheet         0
source_row_number    0
source_record_id     0
dtype: int64

PASS: No provenance metadata is missing.


## 19. Verify the Cold Storage comments field

In [21]:
coldstorage_stage1b = stage1b_dataframes["coldstorage"]

coldstorage_comments_non_null = (
    coldstorage_stage1b["comments_raw"]
    .notna()
    .sum()
)

print(
    "Cold Storage non-null comments_raw:",
    coldstorage_comments_non_null
)

assert coldstorage_comments_non_null == 0

print(
    "PASS: Missing Cold Storage comments field "
    "is represented as null."
)


Cold Storage non-null comments_raw: 0
PASS: Missing Cold Storage comments field is represented as null.


## 20. Inspect provenance together with raw source values

In [22]:
columns_to_display = [
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",
    "product_name_raw",
    "brand_raw",
    "specification_raw",
    "product_price_raw",
]

display(
    stage1b_all[
        columns_to_display
    ].head(10)
)


,retailer,source_file,source_sheet,source_row_number,source_record_id,product_name_raw,brand_raw,specification_raw,product_price_raw
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,Sojasun Natural Soya Yoghurt,Sojasun,6 × 100 g,12.00
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,Monoprix Chocolate Custard Dessert Cream 4x125G,Monoprix,4 × 125 g,10.90
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,Top Gourmet Cheng Tng,Top Gourmet,400 g,3.00
3,redmart,redmart_all_products_final.xlsx,Sheet1,5,redmart_00000005,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,La Laitiere,4 × 115 g,10.09
4,redmart,redmart_all_products_final.xlsx,Sheet1,6,redmart_00000006,Monoprix Flan Caramel Vanilla Pudding 4X100G,Monoprix,4 × 100 g,7.80
5,redmart,redmart_all_products_final.xlsx,Sheet1,7,redmart_00000007,Fortune Sweetened Chin Chow 1KG,Fortune,1 kg,2.85
6,redmart,redmart_all_products_final.xlsx,Sheet1,8,redmart_00000008,RedMart Trio Swiss Roll (Singapore Special),RedMart,430 g,10.10
7,redmart,redmart_all_products_final.xlsx,Sheet1,9,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,Arla,200 g,4.30
8,redmart,redmart_all_products_final.xlsx,Sheet1,10,redmart_00000010,Fortune Soya Beancurd Almond 360G,Fortune,360 g,1.86
9,redmart,redmart_all_products_final.xlsx,Sheet1,11,redmart_00000011,Arla Protein Hazelnut Pudding 200G - Lactose Free,Arla,200 g,4.30


## 21. Save retailer-specific Stage 1B checkpoints

These CSV files are derived staging artifacts. The original Excel files in `data/raw` remain unchanged.


In [23]:
for retailer, df in stage1b_dataframes.items():
    output_file = (
        STAGING_DATA_DIR
        / f"{retailer}_stage1b_raw.csv"
    )

    df.to_csv(
        output_file,
        index=False,
        encoding="utf-8"
    )

    print(
        f"Saved {retailer}: "
        f"{output_file}"
    )


Saved redmart: D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\redmart_stage1b_raw.csv
Saved fairprice: D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\fairprice_stage1b_raw.csv
Saved coldstorage: D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\coldstorage_stage1b_raw.csv


## 22. Save the combined Stage 1B checkpoint

In [24]:
combined_output_file = (
    STAGING_DATA_DIR
    / "all_retailers_stage1b_raw.csv"
)

stage1b_all.to_csv(
    combined_output_file,
    index=False,
    encoding="utf-8"
)

print("Saved combined Stage 1B dataset:")
print(combined_output_file)


Saved combined Stage 1B dataset:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1b_raw.csv


## 23. Verify the output files were created

In [25]:
output_files = [
    STAGING_DATA_DIR / "redmart_stage1b_raw.csv",
    STAGING_DATA_DIR / "fairprice_stage1b_raw.csv",
    STAGING_DATA_DIR / "coldstorage_stage1b_raw.csv",
    STAGING_DATA_DIR / "all_retailers_stage1b_raw.csv",
]

for file_path in output_files:
    print(
        file_path.name,
        "->",
        file_path.exists()
    )

    assert file_path.exists()

print("\nPASS: All Stage 1B output files were created.")


redmart_stage1b_raw.csv -> True
fairprice_stage1b_raw.csv -> True
coldstorage_stage1b_raw.csv -> True
all_retailers_stage1b_raw.csv -> True

PASS: All Stage 1B output files were created.


## 24. Final Stage 1B validation

In [26]:
print("=" * 65)
print("STAGE 1B FINAL VALIDATION")
print("=" * 65)

print(
    f"RedMart records      : "
    f"{len(stage1b_dataframes['redmart']):,}"
)

print(
    f"FairPrice records    : "
    f"{len(stage1b_dataframes['fairprice']):,}"
)

print(
    f"Cold Storage records : "
    f"{len(stage1b_dataframes['coldstorage']):,}"
)

print("-" * 65)

print(
    f"Total records        : "
    f"{len(stage1b_all):,}"
)

print(
    f"Unique source IDs    : "
    f"{stage1b_all['source_record_id'].nunique():,}"
)

print(
    f"Duplicate source IDs : "
    f"{stage1b_all['source_record_id'].duplicated().sum():,}"
)

print(
    f"Number of columns    : "
    f"{len(stage1b_all.columns)}"
)

print("=" * 65)

assert len(stage1b_all) == 177_638
assert stage1b_all["source_record_id"].nunique() == 177_638
assert stage1b_all["source_record_id"].duplicated().sum() == 0
assert len(stage1b_all.columns) == 17

print()
print(
    "PASS: Stage 1B source metadata and "
    "record-ID creation succeeded."
)


STAGE 1B FINAL VALIDATION
RedMart records      : 97,779
FairPrice records    : 69,402
Cold Storage records : 10,457
-----------------------------------------------------------------
Total records        : 177,638
Unique source IDs    : 177,638
Duplicate source IDs : 0
Number of columns    : 17

PASS: Stage 1B source metadata and record-ID creation succeeded.


## Stage 1B result

When the final validation passes, Stage 1B has established a traceable provenance layer for all **177,638** raw retailer records.

No substantive product-data normalization has been performed yet.

The next stage is **Stage 1C**, where text, brand, price, stock and offer fields can be normalized while continuing to preserve the raw source values.
